## Concept focus — Generics and reusable type relationships

Generics let you describe relationships between values instead of hard-coding one concrete type. This matters when you want abstractions like containers, protocols, or helper functions to stay reusable without giving up type precision.

```text
Box[T]
  stores one T

Box[int]    -> holds ints
Box[str]    -> holds strings

T links input and output expectations together
```

### How to think about it
Think of a type variable as a promise: “whatever type comes in here is the same type that comes out there.” Generics are most useful when they preserve relationships, not when they merely add extra notation.

### Visual references and further study
- [typing documentation](https://docs.python.org/3/library/typing.html)
- [mypy generics docs](https://mypy.readthedocs.io/en/stable/generics.html)
- [PEP 695 — type parameter syntax](https://peps.python.org/pep-0695/)
- [Pyright docs](https://microsoft.github.io/pyright/)

---

# Module 17 — Typing and Static Analysis

## Exercise 17.3 — Typed abstractions

Run:  mypy --strict ex03_generics.py && python ex03_generics.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The basics, in modern syntax

In [ ]:
def greet(name: str, times: int = 1) -> str: ...

nums: list[int] = []                    # builtin generics, 3.9+
mapping: dict[str, list[int]] = {}
pair: tuple[str, int] = ("a", 1)        # a fixed 2-tuple
row: tuple[int, ...] = (1, 2, 3)        # a variable-length tuple
maybe: str | None = None                # union syntax, 3.10+

Do not write `List[int]`, `Dict[str, int]`, or `Optional[str]` in new code. They
are the pre-3.9 spellings, still valid and now noise. `ruff`'s `UP` rules rewrite
them for you.

**Accept the widest type, return the narrowest.**

In [ ]:
from collections.abc import Iterable, Sequence, Mapping

def total(values: Iterable[float]) -> float: ...     # any iterable works
def process(items: Sequence[str]) -> list[str]: ...  # needs len/indexing
def lookup(config: Mapping[str, int]) -> int: ...    # read-only dict

Taking `Iterable` means a list, tuple, set, generator, or dict-keys view all
work. Taking `list` rejects four of those for no reason. Returning `list` tells
the caller they can index it; returning `Iterable` makes them guess.

And a hard-won rule from Module 14: **if your function iterates its argument
twice, it must take `Sequence`, not `Iterable`.** The type is the contract, and
`Iterable` promises only one pass.

---

## Concept 3. The vocabulary worth knowing

In [ ]:
from typing import Any, Literal, Final, TypeAlias, Self, NoReturn, cast, overload

x: Final = 3.14                          # never reassigned
Mode = Literal["r", "w", "a"]            # exactly these three strings
UserId: TypeAlias = int                  # a readable alias

def fail(msg: str) -> NoReturn:          # never returns normally
    raise RuntimeError(msg)

class Builder:
    def add(self, x: int) -> Self:       # 3.11+: returns THIS subclass
        return self

**`Literal` is underused and excellent.** `mode: Literal["r", "w"]` catches
`mode="rw"` at check time, and a `match` over a `Literal` can be checked for
exhaustiveness.

**`Any` disables checking**, silently and infectiously — every expression
involving an `Any` becomes `Any`. Treat it as a `# type: ignore` with a wider
blast radius. Prefer `object` when you truly do not know: `object` is safe (you
must narrow before using it), whereas `Any` permits everything.

**`cast` does nothing at runtime.** It is an assertion to the checker that you
know better. Each one is a place you have taken responsibility for a bug the
checker can no longer find.

---

## Concept 4. Generics

```text
from collections.abc import Callable

def first[T](items: Sequence[T]) -> T | None:        # PEP 695, 3.12+
    return items[0] if items else None

class Stack[T]:                                       # generic class, 3.12+
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None: self._items.append(item)
    def pop(self) -> T: return self._items.pop()
```


Pre-3.12 syntax, still everywhere:

In [ ]:
from typing import TypeVar, Generic
T = TypeVar("T")

def first(items: Sequence[T]) -> T | None: ...
class Stack(Generic[T]): ...

Constrained and bounded type variables:

In [ ]:
T = TypeVar("T", int, str)          # CONSTRAINED: exactly int or str
N = TypeVar("N", bound=Number)      # BOUND: Number or any subclass

**Variance in one paragraph.** `Sequence[T]` is *covariant*: a
`Sequence[Dog]` is acceptable where a `Sequence[Animal]` is wanted, because you
can only read from it. `list[T]` is *invariant*: a `list[Dog]` is **not**
acceptable where a `list[Animal]` is wanted, because the callee could append a
`Cat` to it and break the caller's assumption. This is why taking `Sequence`
rather than `list` is not only more permissive but also more type-correct.

---

## Concept 8. Static types versus runtime validation

**They solve different problems and you need both.**

| | Static (mypy) | Runtime (Pydantic) |
|---|---|---|
| When | Before running | While running |
| Cost | Zero at runtime | Real, per object |
| Catches | Wrong types in *your* code | Wrong types in *incoming data* |
| Cannot catch | Bad JSON from a client | A bug in a branch never run |

In [ ]:
# a boundary: data you did not create
class UserIn(BaseModel):          # Pydantic: validates and coerces at runtime
    name: str
    age: int

# inside: data you did create
@dataclass(frozen=True)           # dataclass + mypy: checked statically, free
class User:
    name: str
    age: int

**Validate at the boundary, trust inside.** Once `UserIn` has parsed the
request, everything downstream can rely on `age` being an `int`, and mypy will
enforce that it stays one. Module 28 builds on this.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The basics, in modern syntax
- Section 2: `Optional` is not "optional"
- Section 3: The vocabulary worth knowing
- Section 4: Generics
- Section 5: `Protocol`: structural typing
- Section 6: `TypedDict`, `NewType`, `overload`
- Section 7: Running the checkers
- Section 8: Static types versus runtime validation

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Iterable
from dataclasses import dataclass
from typing import Generic, Protocol, TypeVar

T = TypeVar("T")
E = TypeVar("E")
U = TypeVar("U")


# TODO 1 -----------------------------------------------------------------------
# Result[T, E] -- a value or an error, without exceptions.
#
#   Ok(42).map(str)                -> Ok("42")
#   Err("nope").map(str)           -> Err("nope")   (map is skipped)
#   Ok(42).unwrap()                -> 42
#   Err("nope").unwrap()           -> raises
#   Ok(42).unwrap_or(0)            -> 42
#   Ok(2).and_then(safe_divide)    -> chains, short-circuiting on the first Err
#
# Make mypy able to tell Ok from Err after a check -- that is the hard part.
# Look up how to make `if result.is_ok():` narrow the type, and note what it
# costs compared with exceptions.
#
# Then answer: Rust and Go use this style; Python has exceptions. When is
# Result better in Python, and when is it fighting the language?


# TODO 2 -----------------------------------------------------------------------
# A typed Pipeline: Pipeline[A] with .then(f: Callable[[A], B]) -> Pipeline[B]
#
#   Pipeline(5).then(str).then(len).value    # -> 1, and mypy knows it is an int
#
# The type must change as it flows. Getting mypy to track that through three
# stages is the exercise.


# TODO 3 -----------------------------------------------------------------------
# A generic Repository Protocol:
#
#   class Repository(Protocol[T]):
#       def get(self, id: str) -> T | None: ...
#       def save(self, entity: T) -> None: ...
#       def all(self) -> Iterable[T]: ...
#
# Write InMemoryRepository[T] satisfying it, and a function that accepts any
# Repository[User] -- including one backed by a dict, and one backed by a fake.
#
# Then answer: should Protocol[T] be covariant, contravariant, or invariant
# here, and why? Try declaring it covariant and see what mypy says about save().


# TODO 4 -----------------------------------------------------------------------
# A bounded TypeVar: write `largest(items)` that works for anything supporting
# <, and rejects things that do not -- at CHECK time, not runtime.
#   largest([3, 1, 2])         -> 3
#   largest(["b", "a"])        -> "b"
#   largest([object(), object()])  -> mypy error
#
# Hint: you need a Protocol with __lt__, and the TypeVar bound to it. Note the
# subtlety in typing __lt__'s parameter -- and why the standard library's own
# SupportsRichComparison is defined the way it is.

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    ok: Result[int, str] = Ok(42)              # type: ignore[name-defined]
    err: Result[int, str] = Err("nope")        # type: ignore[name-defined]

    assert ok.unwrap() == 42
    assert err.unwrap_or(0) == 0
    assert ok.map(lambda x: x * 2).unwrap() == 84
    assert err.map(lambda x: x * 2).unwrap_or(-1) == -1

    def safe_div(x: int) -> Result[int, str]:  # type: ignore[name-defined]
        return Ok(100 // x) if x else Err("division by zero")  # type: ignore[name-defined]

    assert ok.and_then(safe_div).unwrap() == 2
    assert Ok(0).and_then(safe_div).unwrap_or(-1) == -1        # type: ignore[name-defined]

    assert Pipeline(5).then(str).then(len).value == 1          # type: ignore[name-defined]

    repo = InMemoryRepository[str]()                            # type: ignore[name-defined]
    repo.save("a")
    assert list(repo.all()) == ["a"]

    assert largest([3, 1, 2]) == 3                              # type: ignore[name-defined]
    assert largest(["b", "a"]) == "b"                           # type: ignore[name-defined]

    print("all generic checks passed (now run mypy --strict)")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.